# 04 - Stress Predictions Data Wrangling

Notebook ini memproses dataset `stress_predictions.csv`.

Tabel ini berisi hasil prediksi harian yang terhubung dengan aktivitas harian melalui `activity_id`. Fokus wrangling adalah menjaga validitas tipe data, rentang `stress_score`, kategori `stress_level`, dan relasi ke `daily_activities_clean`.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Mengatur tampilan dataframe agar output notebook lebih mudah dibaca.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Cari root project otomatis
# Mengambil lokasi kerja notebook saat ini.
current_path = Path.cwd().resolve()

# Menelusuri parent folder sampai menemukan root project yang memiliki folder data/raw.
for path in [current_path] + list(current_path.parents):
    if (path / "data" / "raw").exists():
        PROJECT_ROOT = path
        break

# Menentukan folder sumber data raw.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
# Menentukan folder output data hasil cleaning.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# Menentukan folder output report dan validation summary.
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

# Membuat folder processed jika belum tersedia.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# Membuat folder reports jika belum tersedia.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Menampilkan path project untuk memastikan notebook membaca folder yang benar.
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR   :", REPORT_DIR)

PROJECT_ROOT : C:\Data Codingan\student_stress_data_science
RAW_DIR      : C:\Data Codingan\student_stress_data_science\data\raw
PROCESSED_DIR: C:\Data Codingan\student_stress_data_science\data\processed
REPORT_DIR   : C:\Data Codingan\student_stress_data_science\outputs\reports


## 1. Load Dataset

In [ ]:
# memuat dataset dari folder yang sesuai dan menampilkan sampel awal data.
# Membaca file CSV ke dalam dataframe.
stress_predictions = pd.read_csv(RAW_DIR / "stress_predictions.csv")
users_clean = pd.read_csv(PROCESSED_DIR / "users_clean.csv")
daily_clean = pd.read_csv(PROCESSED_DIR / "daily_activities_clean.csv")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
stress_predictions.head()

,id,user_id,activity_id,prediction_date,stress_score,stress_level,created_at
0,1,1,1,2026-01-01,49.60,Medium,2026-01-01 23:58:00
1,2,1,2,2026-01-02,49.54,Medium,2026-01-02 20:43:00
2,3,1,3,2026-01-03,50.63,Medium,2026-01-03 20:49:00
3,4,1,4,2026-01-04,59.36,Medium,2026-01-05 00:56:00
4,5,1,5,2026-01-05,56.98,Medium,2026-01-05 23:09:00


## 2. Assessing Data

Pemeriksaan difokuskan pada duplicate `activity_id`, validitas kategori `stress_level`, rentang `stress_score`, dan missing value pada kolom penting.

In [ ]:
# menampilkan struktur dataframe, tipe data, dan jumlah nilai non-null.
# Menampilkan struktur kolom, tipe data, dan jumlah non-null.
stress_predictions.info()

<class 'pandas.DataFrame'>
RangeIndex: 27000 entries, 0 to 26999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               27000 non-null  int64  
 1   user_id          27000 non-null  int64  
 2   activity_id      27000 non-null  int64  
 3   prediction_date  27000 non-null  str    
 4   stress_score     27000 non-null  float64
 5   stress_level     27000 non-null  str    
 6   created_at       27000 non-null  str    
dtypes: float64(1), int64(3), str(3)
memory usage: 1.4 MB


In [ ]:
# menampilkan ringkasan statistik numerik dan kategorikal.
# Menampilkan ringkasan statistik untuk kolom numerik dan kategorikal.
stress_predictions.describe(include='all')

,id,user_id,activity_id,prediction_date,stress_score,stress_level,created_at
count,27000.00000,27000.000000,27000.00000,27000,27000.000000,27000,27000
unique,NaN,NaN,NaN,90,NaN,6,18991
top,NaN,NaN,NaN,2026-01-01,NaN,Medium,2026-02-02 23:19:00
freq,NaN,NaN,NaN,300,NaN,23509,6
mean,13500.50000,150.500000,13500.50000,NaN,56.036900,NaN,NaN
std,7794.37297,86.603663,7794.37297,NaN,9.551188,NaN,NaN
min,1.00000,1.000000,1.00000,NaN,21.000000,NaN,NaN
25%,6750.75000,75.750000,6750.75000,NaN,49.350000,NaN,NaN
50%,13500.50000,150.500000,13500.50000,NaN,55.400000,NaN,NaN
75%,20250.25000,225.250000,20250.25000,NaN,62.240000,NaN,NaN


In [ ]:
# menilai missing value, duplicate, dan kandidat masalah kualitas data.
# Mengubah kolom ke tipe numerik; nilai yang gagal dikonversi menjadi NaN.
stress_score_numeric = pd.to_numeric(stress_predictions["stress_score"], errors="coerce")
# Membersihkan whitespace dan menstandarkan format teks.
stress_level_standard = stress_predictions["stress_level"].astype(str).str.strip().str.title()

print("Missing value:")
# Menghitung jumlah missing value pada setiap kolom.
print(stress_predictions.isna().sum())

# Mengecek keberadaan data duplicate berdasarkan aturan yang relevan.
print("\nDuplicate activity_id:", stress_predictions["activity_id"].duplicated().sum())
# Melihat variasi nilai unik untuk menilai konsistensi kategori atau format.
print("Stress level unique:", stress_predictions["stress_level"].unique())
# Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
print("Invalid stress_level count:", (~stress_level_standard.isin(["Low", "Medium", "High"])).sum())

print("\nStress score min:", stress_score_numeric.min())
print("Stress score max:", stress_score_numeric.max())

Missing value:
id                 0
user_id            0
activity_id        0
prediction_date    0
stress_score       0
stress_level       0
created_at         0
dtype: int64

Duplicate activity_id: 0
Stress level unique: <StringArray>
['Medium', 'High', 'Low', 'MEDIUM', 'LOW', 'HIGH']
Length: 6, dtype: str
Invalid stress_level count: 0

Stress score min: 21.0
Stress score max: 92.83


## Insight:

Tabel `stress_predictions` harus selaras dengan `daily_activities_clean`. Jika ada prediksi yang mengarah ke `activity_id` yang tidak tersedia pada daily activity bersih, maka prediksi tersebut tidak dapat dipakai dalam alur data yang konsisten.

Tindakan cleaning diarahkan pada standardisasi tipe data, validasi kategori `stress_level`, pembatasan `stress_score` pada rentang 0 sampai 100, dan penyaringan relasi agar hanya prediction yang memiliki pasangan activity valid yang dipertahankan.

## 3. Cleaning Data

Langkah cleaning:

1. Mengubah key dan tanggal ke tipe data yang sesuai.
2. Menstandarkan format kategori `stress_level`.
3. Menghapus baris dengan kolom penting yang tidak valid.
4. Membatasi `stress_score` pada rentang 0 sampai 100.
5. Mempertahankan hanya baris yang memiliki `user_id` dan `activity_id` valid.
6. Jika terdapat lebih dari satu prediction untuk activity yang sama, dipilih record terbaru berdasarkan `created_at`.

In [ ]:
# membuat salinan dataframe lalu menjalankan proses cleaning sesuai hasil assessing.
stress_predictions_clean = stress_predictions.copy()

# Mengubah kolom ke tipe numerik; nilai yang gagal dikonversi menjadi NaN.
stress_predictions_clean["id"] = pd.to_numeric(stress_predictions_clean["id"], errors="coerce")
stress_predictions_clean["user_id"] = pd.to_numeric(stress_predictions_clean["user_id"], errors="coerce")
stress_predictions_clean["activity_id"] = pd.to_numeric(stress_predictions_clean["activity_id"], errors="coerce")
# Mengubah kolom ke tipe datetime; format yang tidak valid menjadi NaT.
stress_predictions_clean["prediction_date"] = pd.to_datetime(stress_predictions_clean["prediction_date"], errors="coerce")
# Mengubah kolom ke tipe numerik; nilai yang gagal dikonversi menjadi NaN.
stress_predictions_clean["stress_score"] = pd.to_numeric(stress_predictions_clean["stress_score"], errors="coerce")
# Membersihkan whitespace dan menstandarkan format teks.
stress_predictions_clean["stress_level"] = stress_predictions_clean["stress_level"].astype(str).str.strip().str.title()
# Mengubah kolom ke tipe datetime; format yang tidak valid menjadi NaT.
stress_predictions_clean["created_at"] = pd.to_datetime(stress_predictions_clean["created_at"], errors="coerce")

# Menghapus baris yang kehilangan kolom kunci atau informasi penting.
stress_predictions_clean = stress_predictions_clean.dropna(
    subset=["id", "user_id", "activity_id", "prediction_date", "stress_score", "stress_level", "created_at"]
)

stress_predictions_clean = stress_predictions_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    stress_predictions_clean["stress_level"].isin(["Low", "Medium", "High"])
]

# Membatasi nilai agar tetap berada pada rentang logis.
stress_predictions_clean["stress_score"] = stress_predictions_clean["stress_score"].clip(0, 100)

valid_user_ids = set(users_clean["id"])
valid_activity_ids = set(daily_clean["id"])

stress_predictions_clean = stress_predictions_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    stress_predictions_clean["user_id"].isin(valid_user_ids)
    & stress_predictions_clean["activity_id"].isin(valid_activity_ids)
]

# Mengurutkan data agar proses deduplikasi atau output lebih stabil.
stress_predictions_clean = stress_predictions_clean.sort_values("created_at")
# Menghapus duplicate sesuai subset key yang ditentukan.
stress_predictions_clean = stress_predictions_clean.drop_duplicates(subset=["activity_id"], keep="last")

stress_predictions_clean["id"] = stress_predictions_clean["id"].astype(int)
stress_predictions_clean["user_id"] = stress_predictions_clean["user_id"].astype(int)
stress_predictions_clean["activity_id"] = stress_predictions_clean["activity_id"].astype(int)
stress_predictions_clean["prediction_date"] = stress_predictions_clean["prediction_date"].dt.strftime("%Y-%m-%d")
stress_predictions_clean["created_at"] = stress_predictions_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

stress_predictions_clean = stress_predictions_clean[
    ["id", "user_id", "activity_id", "prediction_date", "stress_score", "stress_level", "created_at"]
# Mengurutkan data agar proses deduplikasi atau output lebih stabil.
].sort_values("id")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
stress_predictions_clean.head()

,id,user_id,activity_id,prediction_date,stress_score,stress_level,created_at
0,1,1,1,2026-01-01,49.60,Medium,2026-01-01 23:58:00
1,2,1,2,2026-01-02,49.54,Medium,2026-01-02 20:43:00
2,3,1,3,2026-01-03,50.63,Medium,2026-01-03 20:49:00
3,4,1,4,2026-01-04,59.36,Medium,2026-01-05 00:56:00
4,5,1,5,2026-01-05,56.98,Medium,2026-01-05 23:09:00


## Insight Setelah Cleaning:

`stress_predictions_clean` hanya mempertahankan prediksi yang memiliki relasi valid terhadap user dan daily activity. Hal ini penting karena dataset target-based seperti EDA harian atau modelling membutuhkan hubungan yang jelas antara fitur aktivitas dan label stres.

Jika jumlah baris prediction lebih sedikit daripada daily activity, kondisi tersebut bukan masalah untuk tabel master. Pada tahap analisis atau modelling, sinkronisasi dilakukan melalui inner join.

## 4. Validation dan Save Output

In [ ]:
# membuat tabel validasi untuk memastikan hasil cleaning memenuhi aturan kualitas data.
# Membuat dataframe validasi untuk mendokumentasikan hasil pengecekan kualitas data.
validation = pd.DataFrame([
    {"rule": "stress_predictions.activity_id exists in daily_activities", "passed": set(stress_predictions_clean["activity_id"]).issubset(set(daily_clean["id"]))},
    {"rule": "stress_predictions.activity_id unique", "passed": stress_predictions_clean["activity_id"].is_unique},
    {"rule": "stress_score between 0 and 100", "passed": stress_predictions_clean["stress_score"].between(0, 100).all()},
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    {"rule": "stress_level valid", "passed": stress_predictions_clean["stress_level"].isin(["Low", "Medium", "High"]).all()},
])

validation

,rule,passed
0,stress_predictions.activity_id exists in daily...,True
1,stress_predictions.activity_id unique,True
2,stress_score between 0 and 100,True
3,stress_level valid,True


In [ ]:
# menyimpan output hasil cleaning atau report ke folder tujuan.
# Menyimpan dataframe ke file CSV.
stress_predictions_clean.to_csv(PROCESSED_DIR / "stress_predictions_clean.csv", index=False)
validation.to_csv(REPORT_DIR / "stress_predictions_validation.csv", index=False)

print("Saved:", PROCESSED_DIR / "stress_predictions_clean.csv")

Saved: C:\Data Codingan\student_stress_data_science\data\processed\stress_predictions_clean.csv
